In [ ]:
import numpy as np
import os
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, f1_score, accuracy_score)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import subprocess
subprocess.run(['pip', 'install', 'seaborn', 'joblib', 'xgboost'], check=True)
print("✅ Done!")
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR     = r'C:\Users\elent\Downloads\Eye Disease Detection'
FEATURES_DIR = os.path.join(BASE_DIR, 'features')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results')
MODELS_DIR   = os.path.join(BASE_DIR, 'checkpoints')
CLASSES      = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

# Load features
print("📂 Loading features from disk...")
train_features = np.load(os.path.join(FEATURES_DIR, 'train_features.npy'))
train_labels   = np.load(os.path.join(FEATURES_DIR, 'train_labels.npy'))
val_features   = np.load(os.path.join(FEATURES_DIR, 'val_features.npy'))
val_labels     = np.load(os.path.join(FEATURES_DIR, 'val_labels.npy'))
test_features  = np.load(os.path.join(FEATURES_DIR, 'test_features.npy'))
test_labels    = np.load(os.path.join(FEATURES_DIR, 'test_labels.npy'))

print(f"✅ Train : {train_features.shape}")
print(f"✅ Val   : {val_features.shape}")
print(f"✅ Test  : {test_features.shape}")
print(f"\n✅ All features loaded and ready!")

In [ ]:
def evaluate_model(model, features, labels, split_name='Test'):
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='macro')

    # AUC-ROC (one vs rest)
    labels_binarized = label_binarize(labels, classes=[0, 1, 2, 3])
    auc_roc = roc_auc_score(labels_binarized, probabilities, 
                             multi_class='ovr', average='macro')

    print(f"\n📊 {split_name} Results")
    print("-" * 40)
    print(f"  Accuracy    : {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Macro F1    : {macro_f1:.4f}")
    print(f"  AUC-ROC     : {auc_roc:.4f}")
    print(f"\n  Per-class breakdown:")
    print(classification_report(labels, predictions, 
                                target_names=CLASSES, digits=4))

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'auc_roc' : auc_roc,
        'predictions' : predictions,
        'probabilities': probabilities
    }

def plot_confusion_matrix(labels, predictions, title, save_path=None):
    cm = confusion_matrix(labels, predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.5, linecolor='gray')
    plt.title(title, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel('Actual Class', fontsize=12)
    plt.xlabel('Predicted Class', fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved to results folder")

print("✅ Evaluation functions defined")

In [ ]:
from sklearn.svm import SVC
import time

print("🔄 Training SVM with RBF kernel...")
print("   This may take 5-15 minutes on 83k samples...")

# Class weights
class_weights = {0: 0.5610, 1: 1.8392, 2: 2.4224, 3: 0.7931}

start_time = time.time()

svm_model = SVC(
    kernel='rbf',           # Radial Basis Function kernel
    C=10,                   # regularization parameter
    gamma='scale',          # kernel coefficient
    class_weight=class_weights,
    probability=True,       # needed for AUC-ROC calculation
    random_state=42,
    verbose=True
)

svm_model.fit(train_features, train_labels)

elapsed = time.time() - start_time
print(f"\n✅ SVM training complete!")
print(f"   Training time : {elapsed/60:.1f} minutes")
print(f"   Support vectors: {svm_model.n_support_}")

# Save model
joblib.dump(svm_model, os.path.join(MODELS_DIR, 'svm_model.pkl'))
print(f"✅ Model saved to checkpoints folder")